# LX · 01 · Correct and validate the D3 LCMV / Xu dataset (`corr1`; run once, CPU)
The full build at `b38685e` (65/65 participants) failed one validation check, D3-23 [65725, 68225). That window is
safe: it lies exactly at the 3.3 s filter margin, and the validator rebuilt the boundary from seconds rounded to
1 ms in floating point. D3-23 and D3-58 also selected 305.62 s and 309.62 s: an extra final 10 s piece with one
window each, outside the 300 s target. Methodology log entry 53, `docs/lcmv_xu.md` §11.

This notebook:
1. **moves the suite code pin once** to the corrected code, only while no LX experiment has completed work;
2. writes **new** stores `graphsLX_v1_*_corr1` (mode B) and `graphsLX10_v1_*_corr1` (mode A) from the existing
   ones:
   - it removes only the windows outside the corrected 300 s selection (expected: one each for D3-23 and D3-58),
     consistently across every array and table;
   - no beamformer, connectivity or feature is recomputed, and nothing is rebuilt;
   - every step is verified first (EEGLAB headers; the stored windows; the rejection rule re-applied for the two
     changed participants), and nothing is written if a check fails;
3. runs the **full validation of both corrected stores** (exact boundary positions, 300 s cap, split, shapes,
   weights, counts).

The original stores, the 65 participant checkpoints and their LCMV operators are not modified. Experiments then
use only the validated corrected stores. **Do not run `lx_build.py build` again**; it does nothing once the
corrected stores exist. No model is trained here.

In [ ]:
# ===== D3 LCMV / Xu suite setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_LX_CODE = False   # leave False. True only if you were asked to switch this suite to newer code
# The LCMV / Xu suite has its own code pin, <outputs>/D3_LCMV_XU/CODE_COMMIT.txt (all LX notebooks run
# that commit); the D1/D2 and D3 pins are only reported, never changed here.
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--family', 'LX']
     + (['--update-code'] if UPDATE_LX_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)
print('Dataset 3 folder:', PATHS['dataset3']['root'])
LX = OUT + '/D3_LCMV_XU'
print('suite outputs:', LX)

## 1. Move the suite pin to the corrected code (once; only if no LX experiment has completed work)

In [ ]:
import glob
from pathlib import Path
pin = Path(LX) / 'CODE_COMMIT.txt'
pinned = pin.read_text().split()[0]
has_fix = subprocess.run(['git', '-C', '/content/repo', 'cat-file', '-e',
                          f'{pinned}:src/adgraph/lcmvxu/correct.py']).returncode == 0
if not has_fix:
    done = glob.glob(LX + '/experiments_*/**/*.done.json', recursive=True)
    if done:
        raise RuntimeError(f'{len(done)} completed LX experiment unit(s) are pinned to {pinned[:10]}; '
                           'the pin is not moved automatically. Ask before continuing.')
    _run(['git', '-C', '/content/repo', 'checkout', '-q', f'origin/{BRANCH}'])
    _run([sys.executable, 'scripts/colab_setup.py', '--family', 'LX', '--update-code'])
    for m in [m for m in sys.modules if m == 'adgraph' or m.startswith('adgraph.')]:
        del sys.modules[m]
from adgraph.colab import sh
print('suite pin:', pin.read_text().split()[0], '| previous:', pinned[:10])

## 2. Correction (new versioned stores; originals unchanged)

In [ ]:
sh("python scripts/lx_build.py correct")

## 3. Full validation of both corrected stores (all checks must PASS)

In [ ]:
sh("python scripts/lx_build.py validate")

## 4. What changed

In [ ]:
import json
import pandas as pd
pd.set_option('display.max_rows', 100, 'display.width', 250)
run = sorted(glob.glob(LX + '/graphsLX_v*_corr1'))[-1]
summary = json.load(open(run + '/correction.json'))
print('participants changed:', summary['participants_changed'])
display(pd.read_csv(run + '/correction_removed_windows.csv'))
display(pd.read_csv(run + '/correction_participants.csv').query('changed'))
print('mode B windows:', summary['n_windows_mode_B'], '| mode A windows:', summary['n_windows_mode_A'])
print('beamforming repeated:', summary['beamforming_repeated'],
      '| connectivity / features recomputed:', summary['connectivity_or_features_recomputed'])
for d in (run, run.replace('graphsLX_', 'graphsLX10_')):
    v = json.load(open(d + '/validation.json'))
    print(Path(d).name, 'validation passed:', v['passed'], '| windows:', v['n_windows'])